# ZS601: ten-view no-glass depth comparison
Method A: verified 1 cm point-cloud initialization, k=3 RMS × 0.5, nominal opacity 1, no training.
Reuses the authenticated v006 Colab runtime and pinned CUDA renderer. Exact input hashes are checked.
The companion CPU z-buffer and PNG backprojection evaluator run locally from the same ten cameras.
Saved depth: uint16 PNG, single-channel camera Z in millimetres, 0 invalid.


In [ ]:
from pathlib import Path
import json,sys,subprocess,torch
ROOT=Path('/content/zs601-depth-methods-v007')
OLD=Path('/content/zs601-mesh-noglass-v006')
assert torch.cuda.is_available()
print(dict(torch=torch.__version__,cuda=torch.version.cuda,gpu=torch.cuda.get_device_name(),
           sigmoid20_float32=float(torch.sigmoid(torch.tensor(20.,device='cuda')).cpu())))
print(json.loads((ROOT/'run_spec.json').read_text()))


In [ ]:
subprocess.run([sys.executable,'-u',str(ROOT/'run_gaussian_depth.py'),
 '--package',str(OLD/'source/gaussian-splatting-lidar-init'),
 '--source-gaussian',str(OLD/'run/full/point_cloud/iteration_0/point_cloud.ply'),
 '--input-cloud',str(OLD/'input/points_mesh_1cm_noglass.ply'),
 '--spec',str(ROOT/'run_spec.json'),'--views',str(ROOT/'selected_views.json'),
 '--output',str(ROOT/'method_a_gaussian')],check=True)


In [ ]:
from PIL import Image
import numpy as np
out=ROOT/'method_a_gaussian'
views=json.loads((ROOT/'selected_views.json').read_text())
assert len(views)==10
for folder in ['images','alpha','masks','depth','depth_mask']:
    assert len(list((out/folder).glob('*.png')))==10
for v in views:
    d=np.array(Image.open(out/'depth'/v['name']))
    m=np.array(Image.open(out/'depth_mask'/v['name']))>0
    assert d.dtype==np.uint16 and np.array_equal(d>0,m)
    assert d.shape==(v['height'],v['width'])
print((out/'initialization.json').read_text())
print((out/'render_summary.json').read_text())
print('TEN_VIEW_GAUSSIAN_DEPTH_PASS_NO_TRAINING')
